SCM.275x - 発展サプライチェーンシステム計画とネットワーク設計
# **輸送問題 (Transportation Problem)**

### *開始する前に、このノートブックのコピーをGoogleドライブに保存してください！*

## **初期化 (Initialization)**

In [1]:
# # 必要に応じてパッケージをインストール

# !pip install gurobipy   # Gurobi最適化ソルバー
# !pip install pandas     # データ分析・操作用ライブラリ Pandas
# !pip install folium     # インタラクティブマップ作成用ライブラリ Folium
# !pip install geopy      # 距離計算・地理データ処理用ライブラリ Geopy

In [2]:
# 必要なパッケージをすべてインポート

import pandas as pd                   # データの操作と分析用
#import gurobipy as grb                # 数理モデルを解くためのGurobi最適化ライブラリ
import mypulp as grb                # 数理モデルを解くための最適化ライブラリ
import folium                         # インタラクティブマップ作成用
import folium.plugins as plugins      # folium用追加プラグイン
from geopy.distance import geodesic   # 2地点間の測地線距離計算用

## **ヘルパー関数 (Helper functions)**

### **地図上へのノードのプロット**

In [3]:
# foliumを使って地図上にノードをプロットする関数を定義

def plot_nodes(map,                         # ノードをプロットするFoliumマップオブジェクト
               nodes,                       # ノードオブジェクトの辞書（各ノードは緯度や経度などの属性を含む）
               icon,                        # 地図上のマーカーに使用するアイコン記号
               color,                       # マーカーアイコンの色
               background_color,            # マーカーアイコンの背景色
               ):

    # 辞書内の各ノードをループ処理
    for node in nodes.values():

        # foliumマーカーを作成
        marker = folium.Marker(
            location=[node.lat, node.lon],              # マーカーの位置を設定
            popup = (node.ID + "-" + node.name),        # ノードIDと名前のポップアップを作成
            icon=plugins.BeautifyIcon(                  # マーカーアイコンを作成
                icon=icon,
                icon_shape="circle",
                text_color=color,                       # マーカーの色を設定
                border_color=color,
                background_color=background_color,
            )
        )

        # foliumマーカーを地図に追加
        marker.add_to(map)

### **測地線距離の計算**

In [4]:
# 2地点間の測地線距離を計算する関数を定義

def compute_geodesic_distance(origin,       # 出発地ノードオブジェクト
                              destination,  # 目的地ノードオブジェクト
                              unit='km'):   # 単位（'km' または 'mi'）、デフォルトは 'km'

    # 出発地と目的地から座標（緯度と経度）を抽出
    origin_coordinates = [origin.lat, origin.lon]
    destination_coordinates = [destination.lat, destination.lon]

    # 指定された単位（'km' または 'mi'）に基づいて距離を計算
    if unit == 'km':
        distance = geodesic(origin_coordinates, destination_coordinates).km  # キロメートル単位で距離を計算
    elif unit == 'mi':
        distance = geodesic(origin_coordinates, destination_coordinates).mi  # マイル単位で距離を計算

    return distance  # 計算された距離を返す

### **地図上へのフローのプロット**

In [5]:
# foliumを使って地図上にフローをプロットする関数を定義

def plot_flows(map,                   # フローをプロットするFoliumマップオブジェクト
               vars,                  # 最適化モデルの決定変数の辞書
               nodes,                 # ノードオブジェクトの辞書
               max_width = 30,        # フロー線の最大幅（デフォルトは30）
               color = 'grey',        # フローを表す線の色（デフォルトは灰色）
               opacity = 0.5):        # 線の不透明度（デフォルトは0.5）

    # 線の太さを正規化するためにフローの最大値を検索
    max_val = max([var.X for (node1_key, node2_key), var in vars.items()])

    # フロー決定変数をループ処理（キーはノードペアを表す）
    for (node1_key, node2_key), var in vars.items():

        # 正のフローのみプロット
        if var.X > 0:

            # 線を描画するためのノード座標を取得
            points = [[nodes[node1_key].lat, nodes[node1_key].lon],
                      [nodes[node2_key].lat, nodes[node2_key].lon]]


            # ノード間のフローを表すPolyLineを地図に追加
            folium.PolyLine(points,
                            color=color,                                # 線の色を設定
                            weight=var.X / max_val * max_width,         # フロー値に基づいて線の太さを正規化
                            opacity=opacity,                            # 線の不透明度を設定
                            popup=var.X).add_to(map)                    # 地図のポップアップにフロー値を表示

## **データ設定と前処理**

### **ノード (Nodes)**

#### 入力ファイルの読み込み

In [6]:
# 顧客データを含むファイル
customer_data_file = 'https://raw.githubusercontent.com/scm275/problem_sets_scm275/main/transportation_problem/customers.csv'

# 顧客データをpandas DataFrameに読み込み
customers_df = pd.read_csv(customer_data_file)

# データの確認のため先頭数行を表示
customers_df.head()

,ID,name,lat,lon,demand
0,C01,Concord,43.206898,-71.537994,9000
1,C02,Trenton,40.220596,-74.769913,10000
2,C03,Austin,30.274670,-97.740349,10000
3,C04,Providence,41.830914,-71.414963,9000
4,C05,Montpelier,44.262436,-72.580536,7000


In [7]:
# サプライヤー（供給業者）データを含むファイル
supplier_data_file = 'https://raw.githubusercontent.com/scm275/problem_sets_scm275/main/transportation_problem/suppliers.csv'

# サプライヤーデータをpandas DataFrameに読み込み
suppliers_df = pd.read_csv(supplier_data_file)

# データの確認のため先頭数行を表示
suppliers_df.head()

,ID,name,lat,lon,supply
0,S01,Columbia,34.000343,-81.033211,40000
1,S02,Jefferson City,38.579201,-92.172935,30000
2,S03,Santa Fe,35.682240,-105.939728,20000
3,S04,Richmond,37.538857,-77.433640,32000
4,S05,Carson City,39.163914,-119.766121,20000


#### クラスの定義

In [8]:
# 顧客（Customer）オブジェクトを表すクラス

class Customer():
    def __init__(self, ID, name, lat, lon, demand):
        self.ID = ID              # 顧客ID
        self.name = name          # 顧客名
        self.lat = lat            # 顧客の緯度
        self.lon = lon            # 顧客の経度
        self.demand = demand      # 顧客の需要量

In [9]:
# サプライヤー（Supplier）オブジェクトを表すクラス

class Supplier():
    def __init__(self, ID, name, lat, lon, supply):
        self.ID = ID            # サプライヤーID
        self.name = name        # サプライヤー名
        self.lat = lat          # サプライヤーの緯度
        self.lon = lon          # サプライヤーの経度
        self.supply = supply    # サプライヤーの供給可能量

#### ノードオブジェクトの作成

In [10]:
# ノードオブジェクトを格納する空の辞書を初期化
nodes = dict()

In [11]:
# 顧客オブジェクトの辞書を作成
customers = dict()
for i, row in customers_df.iterrows():
    customers[row['ID']] = Customer(ID=row['ID'],           # 顧客ID
                                    name=row['name'],       # 顧客名
                                    lat=row['lat'],         # 顧客の緯度
                                    lon=row['lon'],         # 顧客の経度
                                    demand=row['demand'])   # 顧客の需要量

# 顧客辞書を既存のnodes辞書に統合
nodes = {**nodes, **customers}

In [12]:
# サプライヤーオブジェクトの辞書を作成
suppliers = dict()
for i, row in suppliers_df.iterrows():
    suppliers[row['ID']] = Supplier(ID=row['ID'],           # サプライヤーID
                                    name=row['name'],       # サプライヤー名
                                    lat=row['lat'],         # サプライヤーの緯度
                                    lon=row['lon'],         # サプライヤーの経度
                                    supply=row['supply'])   # サプライヤーの供給可能量

# サプライヤー辞書を既存のnodes辞書に統合
nodes = {**customers, **suppliers}

In [13]:
# 総供給量が総需要量以上（または以下）であることを確認

# すべてのサプライヤーからの総供給量を格納する変数を初期化
total_supply = 0
# すべてのサプライヤーオブジェクトをループ処理し、供給量を合計
for x in suppliers.values():
    total_supply += x.supply

# すべての顧客からの総需要量を格納する変数を初期化
total_demand = 0
# すべての顧客オブジェクトをループ処理し、需要量を合計
for x in customers.values():
    total_demand += x.demand

# 計算された総供給量を出力
print('Total Supply:', total_supply)

# 計算された総需要量を出力
print('Total Demand:', total_demand)

Total Supply: 171000
Total Demand: 171000


#### ノードオブジェクトの可視化

In [ ]:
# 米国を中心としたズームレベル5の新しい地図を作成
# 米国を中心としたズームレベル5の新しい地図を作成
# (OpenStreetMapのiframe制限やCartoDBのAPIキー要求を回避するため、Esriのライトグレー背景地図を使用)
map = folium.Map(
    location=[40, -95.0],
    zoom_start=5,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Esri'
)
# 顧客の位置を店舗アイコン、緑色、黄色背景でプロット
plot_nodes(map=map, nodes=customers, icon='store', color='green', background_color='yellow')

# サプライヤーの位置を工場アイコン、オレンジ色、黄色背景でプロット
plot_nodes(map=map, nodes=suppliers, icon='industry', color='orange', background_color='yellow')

# 地図の視覚化を向上させるためのタイルレイヤーを追加 (cartodbpositronテーマ)
#folium.TileLayer('cartodbpositron').add_to(map)

# プロットされたすべてのデータを地図に表示
map

/Users/mikiokubo/Documents/dev/lecture2026/.venv/lib/python3.10/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


### **アーク (Arcs)**

#### アークの距離

In [15]:
# 顧客とサプライヤー間の距離を格納する辞書を作成
distances = dict()
for s, supplier in suppliers.items():                                                                   # サプライヤーをループ処理
    for c, customer in customers.items():                                                               # 顧客をループ処理
        distances[s, c] = compute_geodesic_distance(origin=customer, destination=supplier, unit='km')   # キロメートル単位で距離を計算して格納

#### アークのコスト

In [16]:
cost_unit_km = 1.2  # 1kmあたりの単位輸送コスト

# 顧客とサプライヤー間の単位コストを格納する辞書を作成
unit_cost = dict()
for s, supplier in suppliers.items():                                                           # サプライヤーをループ処理
    for c, customer in customers.items():                                                       # 顧客をループ処理
        unit_cost[s, c] = distances[s, c] * cost_unit_km                                        # 距離にkmあたりコストを掛けて単位コストを計算

## **最適化モデル**

### **最適化モデルの作成と求解**

In [17]:
# 新しいGurobiモデルを作成
model = grb.Model("Transportation Problem")

# 決定変数の作成

# サプライヤーと顧客間のフローを表す決定変数
flow_vars = dict()
for s, supplier in suppliers.items():
    for c, customer in customers.items():
        flow_vars[s, c] = model.addVar(vtype=grb.GRB.CONTINUOUS,
                                       name="flow_vars_{0}_{1}".format(s, c))

# 目的関数式の作成
total_cost = grb.quicksum(unit_cost[s, c] * flow_vars[s, c]
                         for s, supplier in suppliers.items()
                         for c, customer in customers.items())

# 目的関数の設定
model.setObjective(total_cost, grb.GRB.MINIMIZE)

# 需要制約の追加
for c, customer in customers.items():
    model.addConstr(grb.quicksum(flow_vars[s, c] for s, supplier in suppliers.items()) == customer.demand)

# 供給制約の追加
for s, supplier in suppliers.items():
    model.addConstr(grb.quicksum(flow_vars[s, c] for c, customer in customers.items()) <= supplier.supply)

# モデルの求解
model.optimize()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/mikiokubo/Documents/dev/lecture2026/.venv/lib/python3.10/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/69/5y96sdc94jxf6khgc8mlmxrr0000gn/T/9ff929ca592d445b92bc4a33283a971c-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/69/5y96sdc94jxf6khgc8mlmxrr0000gn/T/9ff929ca592d445b92bc4a33283a971c-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 31 COLUMNS
At line 392 RHS
At line 419 BOUNDS
At line 540 ENDATA
Problem MODEL has 26 rows, 120 columns and 240 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 26 (0) rows, 120 (0) columns and 240 (0) elements
0  Obj 0 Primal inf 171000 (20)
30  Obj 1.4374377e+08
Optimal - objective value 1.4374377e+08
Optimal objective 143743768.1 - 30 iterations time 0.002
Option for printingOptions changed from normal to all
Total time (CPU secon

/Users/mikiokubo/Documents/dev/lecture2026/.venv/lib/python3.10/site-packages/pulp/pulp.py:1298: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


### **最適化モデルの別定式化**

In [18]:
# 新しいGurobiモデルを作成
model = grb.Model("Transportation Problem")

# 決定変数の作成

# サプライヤーと顧客間のフローを表す決定変数
flow_vars = {(s, c): model.addVar(vtype=grb.GRB.CONTINUOUS,
                                  name="flow_vars_{0}_{1}".format(s, c))
             for s in suppliers.keys()
             for c in customers.keys()
             }

# 目的関数式の作成
total_cost = grb.quicksum(flow_vars[s, c] * unit_cost[s, c]
                         for s in suppliers.keys() for c in customers.keys())

# 目的関数の設定
model.setObjective(total_cost, grb.GRB.MINIMIZE)

# 需要制約の追加
demand_constraints = {c: model.addConstr(grb.quicksum(flow_vars[s, c] for s in suppliers.keys()) == customers[c].demand)
                      for c in customers.keys()
                      }

# 供給制約の追加
supply_constraints = {s: model.addConstr(grb.quicksum(flow_vars[s, c] for c in customers.keys()) <= suppliers[s].supply)
                      for s in suppliers.keys()
                      }

# モデルの求解
model.optimize()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/mikiokubo/Documents/dev/lecture2026/.venv/lib/python3.10/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/69/5y96sdc94jxf6khgc8mlmxrr0000gn/T/e963488b74424ff494754135c1ccdeab-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/69/5y96sdc94jxf6khgc8mlmxrr0000gn/T/e963488b74424ff494754135c1ccdeab-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 31 COLUMNS
At line 392 RHS
At line 419 BOUNDS
At line 540 ENDATA
Problem MODEL has 26 rows, 120 columns and 240 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 26 (0) rows, 120 (0) columns and 240 (0) elements
0  Obj 0 Primal inf 171000 (20)
30  Obj 1.4374377e+08
Optimal - objective value 1.4374377e+08
Optimal objective 143743768.1 - 30 iterations time 0.002
Option for printingOptions changed from normal to all
Total time (CPU secon

## **解の可視化と分析**

In [19]:
# 地図上での顧客とサプライヤーの位置の可視化

# 米国を中心としたズームレベル5の新しい地図を作成
map = folium.Map([40, -95.0], zoom_start=5)

# 顧客の位置を店舗アイコン、緑色、黄色背景でプロット
plot_nodes(map=map, nodes=customers, icon='store', background_color='yellow', color='green')

# サプライヤーの位置を工場アイコン、オレンジ色、黄色背景でプロット
plot_nodes(map=map, nodes=suppliers, icon='industry', background_color='yellow', color='orange')

# サプライヤーと顧客間のフローを最大線幅20でプロット
plot_flows(map=map, max_width=20, vars=flow_vars, nodes=nodes)

# 地図の視覚化を向上させるためのタイルレイヤーを追加 (cartodbpositronテーマ)
#folium.TileLayer('cartodbpositron').add_to(map)

# プロットされたすべてのデータを地図に表示
map